# DVC부터 release manifest까지 하나의 모델 기록 추적하기

## 이번 질문

Git은 코드 변경을, DVC는 데이터와 파이프라인 상태를, MLflow Run은 한 번의 학습 조건과 결과를, 모델 묶음은 실행 파일을 기록합니다. 이 노트북은 흩어진 네 기록이 `release-manifest.json`에서 Candidate B 하나로 이어지는지 순서대로 확인합니다.

공식 Run은 과거 release evidence이고 학생 Run은 같은 구조를 직접 관찰하기 위한 새 개발 실행입니다. 공식 Run과 학생 Run은 서로 다른 실행이며, 학생 Run으로 공식 승인이나 봉인 평가를 바꾸지 않습니다.

## 먼저 예상

Candidate B의 모델 Run ID, 최종 평가 Run ID, 모델 파일 SHA-256이 같은 값인지 먼저 예상합니다. Git commit, DVC revision, MLflow Run ID, 파일 SHA-256 가운데 서로 대신할 수 있는 식별값이 있는지도 적습니다.

## 실행과 관측

### 1. 연결에 필요한 선언을 한 번에 연다

먼저 DVC 분할, 모델 생성, release freeze, 최종 승인, 직렬화 검증 문서를 읽습니다. 각 파일의 역할은 다르며, 뒤 셀에서 Candidate B를 공통 열쇠로 연결합니다.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import yaml

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir()
)
paths = {
    "dvc_split": ROOT / "docs/evidence/data-v2/split-revision.json",
    "model_bootstrap": ROOT / "docs/evidence/model-v2/model-bootstrap.json",
    "release_freeze": ROOT / "docs/evidence/model-v2/release-freeze.json",
    "canonical": ROOT / "docs/evidence/model-v2/canonical-benchmark.json",
    "release_manifest": ROOT / "docs/evidence/model-v2/release-manifest.json",
    "bundle_verification": ROOT
    / "docs/evidence/model-v2/serialized-bundle-verification.json",
    "profiles": ROOT / "configs/model-v2/profiles.yaml",
}
documents = {
    name: (
        yaml.safe_load(path.read_text(encoding="utf-8"))
        if path.suffix in {".yaml", ".yml"}
        else json.loads(path.read_text(encoding="utf-8"))
    )
    for name, path in paths.items()
}
pd.DataFrame(
    {
        "역할": [
            "DVC 데이터 revision과 역할별 지문",
            "모델 Run과 최초 bundle 지문",
            "봉인 평가 전 고정 상태",
            "봉인 평가 결과",
            "승인 profile과 최종 Run 연결",
            "bundle 직렬화 검증",
            "모델 종류와 임계값 선언",
        ],
        "경로": [path.relative_to(ROOT).as_posix() for path in paths.values()],
    },
    index=list(paths),
)

### 2. Git, DVC, MLflow와 모델 파일을 Candidate B로 연결한다

Git commit은 코드를, DVC revision과 dataset SHA-256은 데이터를, MLflow Run ID는 실행을, model/metadata SHA-256은 파일 내용을 식별합니다. `release-manifest.json`은 이 값을 새 식별자로 합치지 않고 승인된 Candidate B와 연결합니다.

과거 `model-bootstrap.json`의 `git_worktree_dirty`와 현재 release manifest의 historical reconciliation은 그대로 표시합니다. 미확인 이력을 숨긴 채 완전 재현이라고 말하지 않습니다.

In [ ]:
split_revision = documents["dvc_split"]
bootstrap = documents["model_bootstrap"]
release_freeze = documents["release_freeze"]
canonical = documents["canonical"]
release_manifest = documents["release_manifest"]
candidate_bundle = bootstrap["bundles"]["candidate-b"]
sealed_role = split_revision["role_datasets"]["test"]

# 이름이 비슷해도 역할이 다른 식별값을 한 표에서 대조한다.
official_model_run = candidate_bundle["mlflow_run_id"]
official_final_run = release_manifest["approved_model"]["final_mlflow_run_id"]
official_chain = pd.DataFrame(
    [
        {
            "기록": "Git code revision",
            "값": bootstrap["provenance"]["git_commit"],
            "의미": "모델 생성에 사용한 코드 commit",
            "근거": "model-bootstrap.json",
        },
        {
            "기록": "DVC data revision",
            "값": f"{split_revision['revision']} / sealed role {sealed_role['rows']} rows",
            "의미": "데이터 역할과 revision",
            "근거": "split-revision.json",
        },
        {
            "기록": "sealed dataset SHA-256",
            "값": sealed_role["sha256"],
            "의미": "봉인 평가 데이터 파일 내용",
            "근거": "split-revision.json",
        },
        {
            "기록": "official model Run",
            "값": official_model_run,
            "의미": "Candidate B 모델 생성 실행",
            "근거": "model-bootstrap.json",
        },
        {
            "기록": "model.joblib SHA-256",
            "값": candidate_bundle["model_sha256"],
            "의미": "실행 모델 파일 내용",
            "근거": "model-bootstrap.json",
        },
        {
            "기록": "metadata.json SHA-256",
            "값": candidate_bundle["metadata_sha256"],
            "의미": "외부 설명 파일 내용",
            "근거": "model-bootstrap.json",
        },
        {
            "기록": "official final Run",
            "값": official_final_run,
            "의미": "봉인 평가 실행",
            "근거": "release-manifest.json",
        },
        {
            "기록": "approved profile",
            "값": release_manifest["approved_profile"],
            "의미": "최종 승인된 모델 종류",
            "근거": "release-manifest.json",
        },
    ]
).set_index("기록")
official_chain

### 3. baseline과 Candidate B의 실행 파일 묶음을 구분한다

하나의 모델 묶음은 `model.joblib`과 `metadata.json` 두 파일입니다. 두 파일은 서로 다른 SHA-256을 가지며, 공통 입력 항목 규약의 SHA-256도 별도로 유지합니다. 지문이 다르면 이름이 같아도 다른 실행 파일 묶음으로 조사합니다.

In [ ]:
bundle_verification = documents["bundle_verification"]
feature_contract_sha256 = release_manifest["approved_model"][
    "feature_contract_sha256"
]
bundle_rows = []
for profile in ("baseline", "candidate-b"):
    verified = bundle_verification["profiles"][profile]
    bundle_rows.append(
        {
            "profile": profile,
            "model.joblib SHA-256": verified["model_sha256"],
            "metadata.json SHA-256": verified["metadata_sha256"],
            "공통 feature contract SHA-256": feature_contract_sha256,
            "canonical metrics 일치": verified["matches_canonical_metrics"],
        }
    )
bundle_comparison = pd.DataFrame(bundle_rows).set_index("profile")
bundle_comparison

### 4. 공식 Candidate B Run의 입력, 설정, 결과와 파일을 한 번에 본다

그림의 한 MLflow Run 화면을 정적 evidence로 복원합니다. 학습 2,900건과 검증 600건, Random Forest와 임계값 0.35, 검증 Recall과 PR-AUC, 모델·설명 파일이 공식 모델 Run 하나에 연결됩니다. 이 셀은 과거 Run을 새 서버에 복제하지 않고 기록된 값을 읽습니다.

In [ ]:
development_path = ROOT / "docs/evidence/model-v2/development-benchmark.json"
development = json.loads(development_path.read_text(encoding="utf-8"))
candidate_profile = next(
    item for item in documents["profiles"]["profiles"] if item["name"] == "candidate-b"
)
candidate_evaluation = next(
    item for item in development["profiles"] if item["profile"] == "candidate-b"
)
official_run_view = pd.DataFrame(
    {
        "값": [
            split_revision["role_datasets"]["train"]["rows"],
            split_revision["role_datasets"]["valid"]["rows"],
            candidate_profile["kind"],
            candidate_profile["threshold"],
            candidate_evaluation["metrics"]["recall"],
            candidate_evaluation["metrics"]["pr_auc"],
            official_model_run,
            candidate_bundle["model_path"],
            candidate_bundle["metadata_path"],
        ]
    },
    index=[
        "train rows",
        "valid rows",
        "model kind",
        "threshold",
        "valid recall",
        "valid PR-AUC",
        "official model Run",
        "model artifact",
        "metadata artifact",
    ],
)
official_run_view

### 5. Candidate B 학생 Run에서 조건, 결과와 생성 파일을 함께 조회한다

`labs/run/log_development.py`는 `train`과 `valid`만 사용해 Candidate B를 새로 학습합니다. 교실 MLflow가 준비되면 experiment `student-development-tracking`에 dataset input, 모델 설정, 검증 지표, `bundle/model.joblib`, `bundle/metadata.json`, MLflow model을 한 Run으로 기록합니다.

이 실행은 구조를 직접 관찰하기 위한 학생 Run입니다. Run ID와 파일 SHA-256이 공식 값과 달라도 정상이며, 공식 승인과 봉인 평가를 다시 수행하지 않습니다. 데이터가 준비되지 않았거나 `AIQA_MLFLOW_TRACKING_URI`가 없으면 공식 연결 표까지만 확인하고 막힌 조건을 그대로 표시합니다.

In [ ]:
import os
import subprocess
import sys

required_development_files = [
    ROOT / "data/features.csv",
    ROOT / "data/splits-v2/split-manifest.csv",
    ROOT / "data/splits-v2/train.csv",
    ROOT / "data/splits-v2/valid.csv",
]
missing_development_files = [
    path.relative_to(ROOT).as_posix()
    for path in required_development_files
    if not path.is_file()
]
if missing_development_files:
    student_result = {
        "status": "DATA_NOT_PREPARED",
        "next_action": "uv run python labs/run/prepare_data.py",
        "missing": missing_development_files,
        "student_run_id": None,
    }
else:
    completed = subprocess.run(
        [sys.executable, str(ROOT / "labs/run/log_development.py")],
        cwd=ROOT,
        env=os.environ.copy(),
        check=True,
        capture_output=True,
        text=True,
    )
    student_result = json.loads(completed.stdout)

pd.DataFrame(
    {"값": list(student_result.values())},
    index=list(student_result),
)

In [ ]:
if student_result["status"] == "LOGGED":
    from mlflow import MlflowClient

    tracking_uri = os.environ["AIQA_MLFLOW_TRACKING_URI"].rstrip("/")
    client = MlflowClient(tracking_uri=tracking_uri)
    student_run = client.get_run(student_result["student_run_id"])
    dataset_inputs = list(getattr(student_run.inputs, "dataset_inputs", []))
    artifact_paths = [item.path for item in client.list_artifacts(student_run.info.run_id)]
    live_run_summary = pd.DataFrame(
        {
            "값": [
                student_run.info.run_id,
                student_run.data.tags.get("aiqa.profile"),
                student_run.data.params.get("model_kind"),
                student_run.data.params.get("threshold"),
                student_run.data.metrics.get("valid.recall"),
                student_run.data.metrics.get("valid.pr_auc"),
                len(dataset_inputs),
                ", ".join(artifact_paths),
                student_result["bundle_model_sha256"],
                student_result["bundle_metadata_sha256"],
            ]
        },
        index=[
            "student_run_id",
            "profile",
            "model_kind",
            "threshold",
            "valid.recall",
            "valid.pr_auc",
            "dataset input 수",
            "root artifacts",
            "bundle_model_sha256",
            "bundle_metadata_sha256",
        ],
    )
else:
    live_run_summary = pd.DataFrame(
        {
            "값": [
                student_result["status"],
                student_result.get("next_action", "교실 MLflow 준비를 확인합니다."),
            ]
        },
        index=["학생 Run 상태", "다음 확인"],
    )
live_run_summary

### 6. 화면의 값이 만들어지는 코드를 따라간다

실행 명령에서 시작해 MLflow adapter, bundle application, release provenance 순서로 읽습니다. Appendix는 DVC와 MLflow API 각각의 문법이 더 필요할 때만 참고합니다.

In [ ]:
code_map = pd.DataFrame(
    [
        {
            "순서": 1,
            "경로": "labs/run/log_development.py",
            "읽을 질문": "학생 Candidate B 실행을 어떤 조건으로 조립하는가",
        },
        {
            "순서": 2,
            "경로": "packages/aiqa_model/adapters/mlflow/model.py",
            "읽을 질문": "dataset, parameter, metric, bundle과 model을 어떻게 한 Run에 기록하는가",
        },
        {
            "순서": 3,
            "경로": "apps/model_trainer/application/bundles.py",
            "읽을 질문": "공식 모델 묶음과 MLflow Run ID를 어떻게 연결하는가",
        },
        {
            "순서": 4,
            "경로": "apps/model_trainer/adapters/release_provenance.py",
            "읽을 질문": "Git, DVC와 파일 지문을 freeze와 manifest에 어떻게 고정하는가",
        },
        {
            "순서": 5,
            "경로": "labs/appendix/09_dvc_basics.ipynb",
            "읽을 질문": "DVC 명령과 lock 문법이 더 필요할 때",
        },
        {
            "순서": 6,
            "경로": "labs/appendix/13_mlflow_basics.ipynb",
            "읽을 질문": "MLflow API와 model load가 더 필요할 때",
        },
    ]
).set_index("순서")
code_map

## 해석과 기록

하나의 모델 기록은 하나의 만능 ID가 아니라 역할이 다른 식별값의 연결입니다. Git commit은 코드 상태, DVC와 dataset SHA-256은 데이터 상태, MLflow Run ID는 실행, model/metadata SHA-256은 파일 내용을 가리킵니다. `release-manifest.json`은 승인된 profile, 모델 Run, 최종 Run과 파일 지문을 연결합니다.

학생 Candidate B Run은 이 구조를 직접 조회하기 위한 개발 실행입니다. 공식 Run과 숫자가 같더라도 공식 evidence가 되지 않으며, Run ID와 bundle SHA-256은 새 실행이므로 달라야 합니다.

## 결과 점검

In [ ]:
assert split_revision["revision"] == "v2"
assert sealed_role["rows"] == 400
assert canonical["sealed_test"]["dataset_sha256"] == sealed_role["sha256"]
assert release_manifest["approved_profile"] == "candidate-b"
assert release_manifest["approved_model"]["model_mlflow_run_id"] == official_model_run
assert official_model_run != official_final_run
assert release_manifest["model_bundles"]["candidate-b/model.joblib"] == candidate_bundle[
    "model_sha256"
]
assert release_manifest["model_bundles"]["candidate-b/metadata.json"] == candidate_bundle[
    "metadata_sha256"
]
assert release_freeze["model_bundles"]["candidate-b/model.joblib"] == candidate_bundle[
    "model_sha256"
]
assert release_freeze["sha256"]["feature_contract_path"] == feature_contract_sha256
assert bundle_comparison.loc["baseline", "model.joblib SHA-256"] != bundle_comparison.loc[
    "candidate-b", "model.joblib SHA-256"
]
assert candidate_profile["kind"] == "random_forest"
assert candidate_profile["threshold"] == 0.35
assert student_result["status"] in {
    "LOGGED",
    "MLFLOW_NOT_RUNNING",
    "DATA_NOT_PREPARED",
}
if student_result["status"] == "LOGGED":
    assert student_result["student_run_id"] != official_model_run
    assert student_result["student_run_id"] != official_final_run
    assert student_result["profile_name"] == "candidate-b"
    assert student_result["model_kind"] == "random_forest"
    assert student_result["threshold"] == 0.35
    assert len(dataset_inputs) == 2
    assert "bundle" in artifact_paths
print("DVC, MLflow, bundle과 release manifest 연결을 확인했습니다.")

## 다음 확인

이제 3장에서 `/v1/model`이 가리키는 profile, version과 model SHA-256을 이 기록의 Candidate B 묶음과 대조합니다. DVC 명령 자체가 더 필요하면 `labs/appendix/09_dvc_basics.ipynb`, MLflow API와 model load가 더 필요하면 `labs/appendix/13_mlflow_basics.ipynb`만 참고합니다.